# 1. Twitter Sentimental Analysis

**Câu 1 (2,0 điểm): Thu thập và phân tích dữ liệu:**
- Phân tích bộ dữ liệu được giao hoặc thu thập dữ liệu dựa trên các API (1đ),
- Thống kê mô tả dữ liệu (1đ).

**Câu 2 (3,0 điểm): Thiết kế mô hình học máy:**
- Làm sạch dữ liệu và import dữ liệu lên Google Colab hoặc Python(1đ)
- Chuẩn hóa và cân bằng dữ liệu (nếu cần thiết) (0.5đ)
- Xây dựng các mô hình dữ liệu, các tập dữ liệu huấn luyện, các tiêu chí đánh giá.(1,5đ)

**Câu 3 (3,0 điểm): So sánh và lựa chọn mô hình học máy:**
- So sánh hiệu quả của các mô hình khác nhau trên bộ dữ liệu được giao (1.5đ)
- Điều chỉnh các tham số để tối ưu hóa mô hình (0.5đ)
- Chạy thử mô hình trong thực tế.(1.5đ)

**Câu 4 (2,0 điểm): Thiết kế một ứng dụng sử dụng mô hình được chọn:**
- Thiết kế giao diện ứng dụng (1,0đ),
- Trình bày báo cáo và trả lời câu hỏi (1,0đ)

In [1]:
# === THIẾT LẬP MÔI TRƯỜNG & NẠP THƯ VIỆN DỰ ÁN ===
import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
REPO_URL = "https://github.com/nhungoc-1/DoAnKHDL_DeTai_Tweet-Hate.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("[SETUP] Đang khởi tạo môi trường Google Colab...", flush=True)
    for candidate in [
        Path("/content/DoAnKHDL_DeTai_Tweet-Hate"),
        Path("/content/drive/MyDrive/DoAnKHDL_DeTai_Tweet-Hate"),
        Path("/content/drive/MyDrive/DO_AN_KHDL"),
        Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
    ]:
        if (candidate / "config.yaml").exists():
            PROJECT_DIR = candidate
            subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)
            break
    else:
        PROJECT_DIR = Path("/content/DoAnKHDL_DeTai_Tweet-Hate")
        print(f"[SETUP] Đang tải dự án từ GitHub vào {PROJECT_DIR} (mất ~3 giây)...", flush=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
    print("[SETUP] Đang kiểm tra & cài đặt các thư viện bổ sung (mất ~5-10 giây)...", flush=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "imbalanced-learn", "emoji", "kagglehub", "datasets",
        "pyvi", "langdetect", "python-docx", "python-pptx", "wordcloud"
    ], check=False)
    import nltk
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.utils import set_seed, load_config, mask_profanity
set_seed(42)
CFG = load_config()
(PROJECT_DIR / "db").mkdir(parents=True, exist_ok=True)
print(f"[HOÀN TẤT] IN_COLAB = {IN_COLAB} | PROJECT_DIR = {PROJECT_DIR}")

[SETUP] IN_COLAB = False
[SETUP] PROJECT_DIR = D:\DO_AN_KHDL


In [2]:
import sklearn
import pandas as pd
import numpy as np
import torch

print("=== THÔNG TIN MÔI TRƯỜNG & PHẦN CỨNG ===")
print(f"Python version      : {sys.version.split()[0]}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version      : {pd.__version__}")
print(f"numpy version       : {np.__version__}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA GPU available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
print("\n=== CẤU HÌNH DỰ ÁN (config.yaml) ===")
print(CFG)
print("\n=== KIỂM TRA HÀM CHE TỪ TỤC (mask_profanity) ===")
sample_text = "This racist bigot called her a bitch and used the word fuck."
print("Trước khi che:", mask_profanity(sample_text))

=== THÔNG TIN MÔI TRƯỜNG & PHẦN CỨNG ===
Python version      : 3.14.6
scikit-learn version: 1.9.1
pandas version      : 3.1.1
numpy version       : 2.5.5
PyTorch version     : 2.15.0+cpu
CUDA GPU available  : False

=== CẤU HÌNH DỰ ÁN (config.yaml) ===
{'seed': 42, 'paths': {'raw': 'data/raw', 'interim': 'data/interim', 'processed': 'data/processed', 'splits': 'data/splits', 'real_world': 'data/real_world', 'reports': 'reports', 'models': 'models', 'db': 'db/tweets.db'}, 'split': {'train': 0.7, 'val': 0.15, 'test': 0.15}, 'tfidf': {'ngram_range': [1, 2], 'min_df': 3, 'max_df': 0.9, 'max_features': 50000, 'sublinear_tf': True}, 'cv': {'folds': 5}, 'primary_metric': 'f1_pos', 'balance': 'class_weight'}

=== KIỂM TRA HÀM CHE TỪ TỤC (mask_profanity) ===
Trước khi che: This racist bigot called her a b***h and used the word f**k.


### Nhận xét 
- Cấu trúc thư mục dự án, tệp cấu hình `config.yaml`, danh sách thư viện `requirements.txt` và mô-đun tiện ích `src/utils.py` đã được khởi tạo đầy đủ.
- Ô khởi tạo môi trường hoạt động ổn định, tự động nhận diện môi trường Google Colab hoặc máy Local mà không cần sửa đường dẫn thủ công.
- Seed ngẫu nhiên (`seed = 42`) và hàm che từ nhạy cảm `mask_profanity` đã sẵn sàng cho toàn bộ các giai đoạn toàn bộ quy trình.